# Module 5.1: What Stays on the Raspberry Pi? Workload Split Strategy & Pi-hole v6 DNS Server

## Homelab Expansion (Chapter 2) - Step 5: Core Raspberry Pi Infrastructure Services

## 🎯 Learning Goals

By completing this laboratory module, you will be able to:

1. Articulate which homelab services belong on an ultra-low-power, 24/7 ARM node (`homelab-pi`) versus a high-power x86/NVIDIA GPU node (`homelab-gpu`).
2. Understand **Fate Sharing / Blast Radius Isolation**: why network-critical services like DNS, VPN routing, and uptime monitoring must never live on a heavy GPU compute worker.
3. Deploy **Pi-hole v6** via Docker Compose on `homelab-pi` using modern `FTLCONF_` configuration variables.
4. Configure local LAN DNS resolution and **Tailscale Global Split-DNS** so every phone and laptop on your Tailnet gets automatic ad-blocking and tracker filtering anywhere in the world.

## 5.1.1 Conceptual Background: What Services Make Sense to Keep on the Raspberry Pi (and Why)?

Now that you have a powerful 6-core/8-core ROG Zephyrus with 16GB–32GB RAM and an NVIDIA GPU sitting next to your 4GB Raspberry Pi 4, a natural question arises: *"Should I just move everything to the laptop and retire the Raspberry Pi?"*

**Absolutely not!** In professional systems engineering, separating **Control-Plane / Network-Core Infrastructure (`homelab-pi`)** from **Compute-Intensive Worker Nodes (`homelab-gpu`)** is the single smartest architectural upgrade you can make:

### 1. Why the Raspberry Pi Is the Ideal "Tier-0 Core Infrastructure" Node
1. **Zero Fate Sharing (Blast Radius Isolation):** Imagine you are experimenting with a new local LLM (`Ollama`), running a massive CUDA video batch job, or updating NVIDIA kernel drivers (`nvidia.ko`) on `homelab-gpu`—requiring a system reboot. If your home's primary DNS server (`Pi-hole`) lived on that laptop, **every device in your house would lose internet DNS resolution** every time you rebooted or tweaked the GPU node!
2. **Ultra-Low 4.5W Power Footprint:** The Raspberry Pi 4 has zero moving parts (fanless), sips ~4.5W of power (running for hours on a small UPS during a power outage), and now has **~2.5 GB of free RAM** because we offloaded `immich-machine-learning` to the laptop in Module 4.2.
3. **Dedicated High-Endurance NVMe NAS Storage:** Your WD Blue SN5000 SSD mounted at `/mnt/homelab_storage` already holds your persistent data and swapfile safely.

### 2. The Definitive Multi-Node Service Allocation Matrix

| Service | Assigned Node | RAM Usage | Why It Belongs Here |
| :--- | :---: | :---: | :--- |
| **Pi-hole v6 (Network-Wide DNS Sinkhole & Local DNS)** | **Node 1 (`homelab-pi`)** | ~65 MB | Tier-0 network dependency. Must stay online 24/7/365 with sub-millisecond response even when the GPU node is rebooting or powered down. |
| **Tailscale Subnet Router & Global DNS Endpoint** | **Node 1 (`homelab-pi`)** | ~35 MB | Provides an always-on entry point into your home LAN (`10.0.0.0/24`) from anywhere in the world. |
| **Uptime Kuma & Watchtower (Monitoring & Alerts)** | **Node 1 (`homelab-pi`)** | ~110 MB | An external observer must never live on the heavy compute machine it is monitoring—if `homelab-gpu` hangs, `homelab-pi` alerts you immediately! |
| **Vaultwarden (Self-Hosted Bitwarden Password Vault) / WireGuard** | **Node 1 (`homelab-pi`)** | ~45 MB | Lightweight Rust/SQLite microservice; needs 99.99% uptime and zero GPU compute. |
| **Immich Core (`immich-server`, `postgres`, `redis`) + NVMe Library** | **Node 1 (`homelab-pi`)** | ~950 MB | Keeps photo browsing and mobile photo backups online 24/7 at 5W idle power. |
| **Immich Machine Learning (`release-cuda`)** | **Node 2 (`homelab-gpu`)** | ~1.5 GB VRAM | Needs 1,500+ NVIDIA CUDA cores and GDDR6 VRAM for 40x faster CLIP & face recognition. |
| **Hardware Media Transcoding (Jellyfin / Plex NVENC)** | **Node 2 (`homelab-gpu`)** | ~800 MB | Uses the Turing NVENC silicon encoder to transcode 4K HEVC video streams in hardware. |
| **Local AI / LLMs (Ollama + Open WebUI)** | **Node 2 (`homelab-gpu`)** | 4–8 GB VRAM | Runs `Llama-3.2` / `Qwen-2.5-Coder` at 40+ tokens/sec directly on the NVIDIA GPU. |

## 5.1.2 Step-by-Step Laboratory Guide: Installing Pi-hole v6 on `homelab-pi`

### Task 5.1.1: Verify Port 53 Availability on `homelab-pi`
DNS operates on **TCP and UDP Port `53`**. Before deploying Pi-hole on `homelab-pi`, connect via `ssh homelab-pi` and verify that no local stub resolver (such as `systemd-resolved`) is occupying port `53`:
```bash
# Check if anything is currently listening on port 53 on homelab-pi
sudo ss -tulpn | grep ":53 " || echo "Port 53 is free and ready for Pi-hole!"
```
*(Note: If `systemd-resolved` is listed on `127.0.0.53:53`, disable its stub listener with `sudo sed -r -i.orig 's/#?DNSStubListener=yes/DNSStubListener=no/g' /etc/systemd/resolved.conf && sudo systemctl restart systemd-resolved`).*

### Task 5.1.2: Create the Pi-hole v6 Docker Compose Stack on the External NVMe SSD
To protect the Raspberry Pi's MicroSD card from continuous SQLite DNS query-database writes (`pihole-FTL.db`), we store Pi-hole's persistent configuration volumes on our external NVMe SSD (`/mnt/homelab_storage/docker_volumes/pihole`):

```bash
# 1. Create stack directory and persistent SSD volume directories on homelab-pi
mkdir -p ~/homelab/stacks/pihole
mkdir -p /mnt/homelab_storage/docker_volumes/pihole/etc-pihole
mkdir -p /mnt/homelab_storage/docker_volumes/pihole/etc-dnsmasq.d
cd ~/homelab/stacks/pihole
```

Create a `.env` file to hold your Pi-hole v6 administrator password (remember from our privacy rule: replace the placeholder with your own strong password locally):
```bash
cat << 'EOT' > ~/homelab/stacks/pihole/.env
PIHOLE_PASSWORD=[Write your password here]
EOT
chmod 600 ~/homelab/stacks/pihole/.env
```

Now create the **Pi-hole v6** `docker-compose.yml` file (note that Pi-hole v6 unified its web server inside `pihole-FTL` and uses `FTLCONF_webserver_api_password` and `FTLCONF_dns_listeningMode=all` instead of legacy v5 variables):
```bash
cat << 'EOT' > ~/homelab/stacks/pihole/docker-compose.yml
name: pihole

services:
  pihole:
    container_name: pihole
    image: pihole/pihole:latest
    ports:
      - "53:53/tcp"
      - "53:53/udp"
      - "8080:80/tcp"
    environment:
      TZ: "America/New_York"
      # Pi-hole v6 unified FTL configuration variables:
      FTLCONF_webserver_api_password: "${PIHOLE_PASSWORD}"
      FTLCONF_dns_upstreams: "1.1.1.1;1.0.0.1;9.9.9.9"
      # 'all' permits queries from both local LAN (10.0.0.x) and Tailscale (100.x.y.z)
      FTLCONF_dns_listeningMode: "all"
    volumes:
      - /mnt/homelab_storage/docker_volumes/pihole/etc-pihole:/etc/pihole
      - /mnt/homelab_storage/docker_volumes/pihole/etc-dnsmasq.d:/etc/dnsmasq.d
    cap_add:
      - NET_ADMIN
    restart: unless-stopped
EOT
```
*(Why port `8080:80`? Mapping the Pi-hole web UI to host port `8080` keeps port `80`/`443` free on `homelab-pi` in case you later add a reverse proxy like Caddy or Nginx).*

Launch the Pi-hole v6 stack:
```bash
cd ~/homelab/stacks/pihole
docker compose up -d
docker compose ps
```

### Task 5.1.3: Verify DNS Resolution, Custom Local DNS Records & Web Dashboard
1. Test DNS resolution directly against `homelab-pi` (`10.0.0.200`) from your terminal:
   ```bash
   # 1. Verify normal domain resolution works (<20ms)
   nslookup github.com 10.0.0.200

   # 2. Verify a known advertising/tracking domain is sinkholed to 0.0.0.0!
   nslookup doubleclick.net 10.0.0.200
   ```
2. Open the **Pi-hole v6 Web Admin Interface** in your browser:
   ```text
   http://10.0.0.200:8080/admin
   ```
3. Log in with your password and add **Local DNS Records** (**Settings $\rightarrow$ Local DNS Records**) for your two homelab nodes so every device on your network can resolve them by clean names:
   - `homelab-pi.lan` $\rightarrow$ `10.0.0.200`
   - `homelab-gpu.lan` $\rightarrow$ `10.0.0.201`
4. **Integrate with Your Home Network & Tailscale Mesh:**
   - **Home LAN:** In your home router's DHCP settings, set the **Primary DNS Server** to `10.0.0.200` so every smart TV, laptop, and phone on home Wi-Fi uses Pi-hole automatically.
   - **Remote Tailscale Ad-Blocking Everywhere:** In the [Tailscale Admin Console $\rightarrow$ DNS tab](https://login.tailscale.com/admin/dns), click **Add nameserver $\rightarrow$ Custom**, enter your Raspberry Pi's `100.x.y.z` Tailscale IP, and toggle **Override local DNS** ON. Now your smartphone blocks ads over 5G cellular data anywhere in the world through your Raspberry Pi!

### 🖼️ Task 5.1.3 Verification: Pi-hole v6 Dashboard & Sinkholed Query Verification

**Verification Objective:** Capture a screenshot of your Pi-hole v6 Web Dashboard (`http://10.0.0.200:8080/admin`) showing active DNS queries blocked and your terminal `nslookup doubleclick.net 10.0.0.200` returning `0.0.0.0`.

---

> ### 📷 User Verification Screenshot Submission Box
> *Attach your milestone verification screenshot below (drag & drop an image or use a Markdown image link):*
> 
> ```markdown
> <!-- Insert your screenshot here: -->
> 
> ```

## 🏁 Module Checkpoint & Reflection

Before proceeding to the next guide, confirm the following checklist:

- [ ] Pi-hole v6 deployed on `homelab-pi` with persistent SQLite storage mapped to `/mnt/homelab_storage/docker_volumes/pihole`.
- [ ] `nslookup doubleclick.net 10.0.0.200` returns `0.0.0.0`.
- [ ] Local DNS records created for `homelab-pi.lan` (`10.0.0.200`) and `homelab-gpu.lan` (`10.0.0.201`).
- [ ] Tailscale / Router DNS pointed to `homelab-pi` for automatic ad-blocking.
- [ ] Verification screenshot captured and attached above.

Proceed to [Module 5.2: Unified Homelab Monitoring with Uptime Kuma](./05_2_homelab_monitoring_uptime_kuma.ipynb).